# 05_sol: Stack Samples to Trace Events

Contains:
- the same scenario as `05_mock`
- one complete reference implementation
- grading tests


In [ ]:
# Chunk overview: Prepare imports, fixtures, and helper scaffolding used by the solution.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Import required modules for this solution step.
from collections import defaultdict
# Import required modules for this solution step.
from typing import Any

# Assign computed data to a named variable for later use.
TRACE_SAMPLES = [
    # Execute this line as part of the solution flow.
    {"ts": 0, "stack": ["main"]},
    # Execute this line as part of the solution flow.
    {"ts": 1, "stack": ["main", "load"]},
    # Execute this line as part of the solution flow.
    {"ts": 3, "stack": ["main", "load", "parse"]},
    # Execute this line as part of the solution flow.
    {"ts": 5, "stack": ["main", "render"]},
# Execute this line as part of the solution flow.
]

# Assign computed data to a named variable for later use.
NOOP_SAMPLES = [
    # Execute this line as part of the solution flow.
    {"ts": 10, "stack": ["main"]},
    # Execute this line as part of the solution flow.
    {"ts": 11, "stack": ["main"]},
# Execute this line as part of the solution flow.
]


In [ ]:
# Chunk overview: Implement the final reference solution in a clean, stepwise way.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Define `samples_to_events` so this step is reusable and testable.
def samples_to_events(samples: list[dict[str, Any]], close_final: bool = False) -> list[dict[str, Any]]:
    # Check this condition to choose the correct branch.
    if not isinstance(samples, list):
        # Raise explicit error to fail fast on invalid state.
        raise ValueError("samples_must_be_list")
    # Check this condition to choose the correct branch.
    if not samples:
        # Return the computed value for the caller.
        return []

    # Assign computed data to a named variable for later use.
    events: list[dict[str, Any]] = []
    # Assign computed data to a named variable for later use.
    prev_ts: int | None = None
    # Assign computed data to a named variable for later use.
    old_stack: list[str] = []

    # Iterate through items to process each element deterministically.
    for sample in samples:
        # Check this condition to choose the correct branch.
        if "ts" not in sample or "stack" not in sample:
            # Raise explicit error to fail fast on invalid state.
            raise ValueError("sample_missing_required_fields")
        # Assign computed data to a named variable for later use.
        ts = sample["ts"]
        # Assign computed data to a named variable for later use.
        new_stack = sample["stack"]
        # Check this condition to choose the correct branch.
        if not isinstance(ts, int):
            # Raise explicit error to fail fast on invalid state.
            raise ValueError("timestamp_must_be_int")
        # Check this condition to choose the correct branch.
        if not isinstance(new_stack, list) or any(not isinstance(frame, str) for frame in new_stack):
            # Raise explicit error to fail fast on invalid state.
            raise ValueError("stack_must_be_list_of_strings")
        # Check this condition to choose the correct branch.
        if prev_ts is not None and ts < prev_ts:
            # Raise explicit error to fail fast on invalid state.
            raise ValueError("timestamps_must_be_non_decreasing")
        # Assign computed data to a named variable for later use.
        prev_ts = ts

        # Assign computed data to a named variable for later use.
        prefix = 0
        # Loop while this condition remains true.
        while prefix < len(old_stack) and prefix < len(new_stack) and old_stack[prefix] == new_stack[prefix]:
            # Assign computed data to a named variable for later use.
            prefix += 1

        # Iterate through items to process each element deterministically.
        for fn in reversed(old_stack[prefix:]):
            # Call this function to perform the next operation.
            events.append({"type": "end", "fn": fn, "ts": ts})
        # Iterate through items to process each element deterministically.
        for fn in new_stack[prefix:]:
            # Call this function to perform the next operation.
            events.append({"type": "start", "fn": fn, "ts": ts})

        # Assign computed data to a named variable for later use.
        old_stack = list(new_stack)

    # Check this condition to choose the correct branch.
    if close_final and samples:
        # Assign computed data to a named variable for later use.
        flush_ts = samples[-1]["ts"] + 1
        # Iterate through items to process each element deterministically.
        for fn in reversed(old_stack):
            # Call this function to perform the next operation.
            events.append({"type": "end", "fn": fn, "ts": flush_ts})

    # Return the computed value for the caller.
    return events


# Define `longest_running_function` so this step is reusable and testable.
def longest_running_function(samples: list[dict[str, Any]]) -> tuple[str, int]:
    # Assign computed data to a named variable for later use.
    events = samples_to_events(samples, close_final=True)
    # Check this condition to choose the correct branch.
    if not events:
        # Raise explicit error to fail fast on invalid state.
        raise ValueError("no_events")

    # Assign computed data to a named variable for later use.
    open_frames: dict[str, int] = {}
    # Assign computed data to a named variable for later use.
    durations: defaultdict[str, int] = defaultdict(int)
    # Iterate through items to process each element deterministically.
    for event in events:
        # Assign computed data to a named variable for later use.
        fn = event["fn"]
        # Assign computed data to a named variable for later use.
        ts = event["ts"]
        # Check this condition to choose the correct branch.
        if event["type"] == "start":
            # Assign computed data to a named variable for later use.
            open_frames[fn] = ts
            # Execute this line as part of the solution flow.
            continue
        # Assign computed data to a named variable for later use.
        start_ts = open_frames.pop(fn, None)
        # Check this condition to choose the correct branch.
        if start_ts is None:
            # Execute this line as part of the solution flow.
            continue
        # Assign computed data to a named variable for later use.
        durations[fn] += ts - start_ts

    # Check this condition to choose the correct branch.
    if not durations:
        # Raise explicit error to fail fast on invalid state.
        raise ValueError("no_durations")

    # Assign computed data to a named variable for later use.
    winner = sorted(durations.items(), key=lambda kv: (-kv[1], kv[0]))[0]
    # Return the computed value for the caller.
    return winner


In [ ]:
# Chunk overview: Run checks that prove the implementation meets the problem contract.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Define `run_exam05_tests` so this step is reusable and testable.
def run_exam05_tests() -> None:
    # Assign computed data to a named variable for later use.
    expected = [
        # Execute this line as part of the solution flow.
        {"type": "start", "fn": "main", "ts": 0},
        # Execute this line as part of the solution flow.
        {"type": "start", "fn": "load", "ts": 1},
        # Execute this line as part of the solution flow.
        {"type": "start", "fn": "parse", "ts": 3},
        # Execute this line as part of the solution flow.
        {"type": "end", "fn": "parse", "ts": 5},
        # Execute this line as part of the solution flow.
        {"type": "end", "fn": "load", "ts": 5},
        # Execute this line as part of the solution flow.
        {"type": "start", "fn": "render", "ts": 5},
        # Execute this line as part of the solution flow.
        {"type": "end", "fn": "render", "ts": 6},
        # Execute this line as part of the solution flow.
        {"type": "end", "fn": "main", "ts": 6},
    # Execute this line as part of the solution flow.
    ]
    # Assign computed data to a named variable for later use.
    events = samples_to_events(TRACE_SAMPLES, close_final=True)
    # Assert expected behavior to validate correctness.
    assert events == expected

    # Existing inline note.
    # unchanged stack should not emit transitions
    # Assign computed data to a named variable for later use.
    events = samples_to_events(NOOP_SAMPLES, close_final=True)
    # Assert expected behavior to validate correctness.
    assert events == [
        # Execute this line as part of the solution flow.
        {"type": "start", "fn": "main", "ts": 10},
        # Execute this line as part of the solution flow.
        {"type": "end", "fn": "main", "ts": 12},
    # Execute this line as part of the solution flow.
    ]

    # Assign computed data to a named variable for later use.
    fn, duration = longest_running_function(TRACE_SAMPLES)
    # Assert expected behavior to validate correctness.
    assert fn == "main"
    # Assert expected behavior to validate correctness.
    assert duration == 6

    # Existing inline note.
    # invalid timestamps
    # Start guarded block to handle potential runtime errors.
    try:
        # Assign computed data to a named variable for later use.
        samples_to_events([{"ts": 2, "stack": ["main"]}, {"ts": 1, "stack": ["main"]}], close_final=True)
        # Raise explicit error to fail fast on invalid state.
        raise AssertionError("Expected timestamp validation failure")
    # Handle expected failure path and keep behavior predictable.
    except ValueError as exc:
        # Assert expected behavior to validate correctness.
        assert "timestamps_must_be_non_decreasing" in str(exc)

    # Call this function to perform the next operation.
    print("05_mock tests passed")


# Call this function to perform the next operation.
run_exam05_tests()
